# InternCircle Data Science & AI Internship: Project 3
## Customer Churn / Sales Trend Analysis Using Python
**Author:** Data Science Intern  
**Dataset:** UCI Online Retail Dataset (541,909 records)  
**Objective:** End-to-end commercial transaction analysis, customer RFM segmentation, sales trend discovery, and strategic business recommendations.


### 1. Import Required Libraries and Configure Environment

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'Arial'

print('Libraries loaded successfully!')


### 2. Data Loading and Initial Inspection

In [ ]:
# Load raw dataset
df = pd.read_csv('ecommerce_data.csv')
print(f'Dataset Dimensions: {df.shape[0]:,} Rows, {df.shape[1]} Columns')
df.head()


In [ ]:
# Dataset info and data types
df.info()


In [ ]:
# Summary statistics
df.describe(include='all')


### 3. Data Cleaning & Validation
- Check missing values
- Drop duplicate records
- Filter out guest records without CustomerID
- Remove invalid quantities (<=0) and prices (<=0)
- Parse dates to datetime


In [ ]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())

# Duplicates
print(f'Duplicate records: {df.duplicated().sum():,}')
df_clean = df.drop_duplicates().copy()

# Drop missing CustomerID for customer-level analytics
df_clean = df_clean.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)

# Filter valid Quantity and UnitPrice
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)].copy()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f'Cleaned dataset rows: {len(df_clean):,} (Retained: {len(df_clean)/len(df)*100:.2f}%)')


### 4. Feature Engineering & Date Extraction

In [ ]:
# Extract calendar features and line-item revenue
df_clean['TotalRevenue'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Day'] = df_clean['InvoiceDate'].dt.day
df_clean['Weekday'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['InvoiceMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)

print('Feature engineering complete. Sample records:')
df_clean[['InvoiceNo', 'CustomerID', 'InvoiceDate', 'Weekday', 'InvoiceMonth', 'Quantity', 'UnitPrice', 'TotalRevenue']].head()


### 5. Customer Analysis & RFM Segmentation

In [ ]:
# Top 10 Customers by Revenue
cust_summary = df_clean.groupby('CustomerID').agg(
    TotalRevenue=('TotalRevenue', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique'),
    Country=('Country', 'first')
).reset_index().sort_values(by='TotalRevenue', ascending=False)

cust_summary.head(10)


In [ ]:
# RFM Customer Segmentation
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalRevenue', 'sum')
).reset_index()

rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4]).astype(int)

def assign_segment(row):
    score = row['R_Score'] + row['F_Score'] + row['M_Score']
    if score >= 10:
        return 'High Value (Champions)'
    elif score >= 8:
        return 'Loyal Customers'
    elif score >= 6:
        return 'Potential Growth'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost / Churned'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

seg_summary = rfm.groupby('Segment').agg(
    CustomerCount=('CustomerID', 'count'),
    TotalRevenue=('Monetary', 'sum'),
    AvgRevenue=('Monetary', 'mean'),
    AvgRecency=('Recency', 'mean'),
    AvgFrequency=('Frequency', 'mean')
).reset_index()

seg_summary['RevenueShare%'] = (seg_summary['TotalRevenue'] / seg_summary['TotalRevenue'].sum()) * 100
seg_summary['CustomerShare%'] = (seg_summary['CustomerCount'] / seg_summary['CustomerCount'].sum()) * 100
seg_summary.sort_values(by='TotalRevenue', ascending=False)


### 6. Sales Trend Analysis

In [ ]:
# Monthly Revenue
monthly = df_clean.groupby('InvoiceMonth').agg(
    TotalRevenue=('TotalRevenue', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique'),
    ActiveCustomers=('CustomerID', 'nunique')
).reset_index()
monthly


### 7. Pivot Tables & Cross-Dimensional Analysis

In [ ]:
# Revenue by Month and Year
pivot_month_year = pd.pivot_table(
    df_clean,
    values='TotalRevenue',
    index='Month',
    columns='Year',
    aggfunc='sum',
    fill_value=0
)
pivot_month_year


### 8. Data Visualizations

In [ ]:
# Monthly Revenue Line Chart
plt.figure(figsize=(11, 5))
plt.plot(monthly['InvoiceMonth'], monthly['TotalRevenue'] / 1000, marker='o', color='#1e3d59', linewidth=2.5)
plt.title('Monthly Sales Revenue (£ in Thousands)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (£k)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
